# Processing and Curation - Fantasy & Paranormal

This notebook executes the reproducible curation from raw JSON to Parquet ready for modeling. The heavy logic lives in `src/process_goodreads.py` and `src/utils/`.

## Cleaning Decisions

- Empty strings are converted to nulls before type casting.
- `rating = 0` in interactions is treated as missing rating, not as negative rating.
- Goodreads dates are parsed with UTC timezone and invalid values are left as null.
- Available review text is `review_text_incomplete`; HTML and whitespace are normalized without inventing missing content.
- Exact duplicates are removed and, for `review_id`, the version with the most recent `date_updated` is kept.
- Outliers are not removed; capped columns at p99 are added for modeling while keeping the originals.

In [ ]:
from pathlib import Path
import os, sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.process_goodreads import process_category
from src.config import CATEGORIES

cfg = CATEGORIES['fantasy_paranormal']

In [ ]:
report = process_category('fantasy_paranormal', chunksize=100_000, bucket_count=256)
report

## Validación de salidas

Se comprueba que los Parquet sean legibles y que las invariantes principales se mantengan.

In [ ]:
books = pd.read_parquet(cfg.processed_dir / 'books_curated.parquet')
interactions = pd.read_parquet(cfg.processed_dir / 'interactions_curated.parquet')

assert books['book_id'].notna().all()
assert interactions['book_id'].notna().all()
assert interactions['rating_clean'].dropna().between(1, 5).all()
assert books['book_id'].duplicated().sum() == 0
assert interactions['review_id'].dropna().duplicated().sum() == 0

books.head(), interactions.head()